# SQL basics (SQLite)

> **Note:** the notebooks in `examples/` are reset every time your server starts, so they are always up to date. To keep your changes, copy this notebook into `work/` (right-click it in the file browser → *Copy*, then paste it in `work/`).

This notebook uses the **xeus-sqlite** kernel: each cell is an **SQL** statement run on an SQLite database.
We'll build a small school database and ask it questions. Run a cell with **Shift + Enter**.

Kernel commands start with `%`: `%CREATE file.db` creates (or opens) a database, `%LOAD file.db rw` opens an existing one.

## 1. Create a database

We keep the database in `/tmp` so it doesn't clutter your files. Run every cell from the top so the database is built in order.

In [1]:
%CREATE /tmp/school.db

In [2]:
DROP TABLE IF EXISTS enrollments;

In [3]:
DROP TABLE IF EXISTS students;

In [4]:
DROP TABLE IF EXISTS courses;

## 2. Tables: `CREATE TABLE`

A **table** has columns, each with a type. A **primary key** identifies each row; a **foreign key** points to a row in another table.

In [5]:
CREATE TABLE students (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL,
    age     INTEGER,
    city    TEXT
);

In [6]:
CREATE TABLE courses (
    id      INTEGER PRIMARY KEY,
    title   TEXT NOT NULL,
    credits INTEGER
);

In [7]:
CREATE TABLE enrollments (
    student_id INTEGER REFERENCES students(id),
    course_id  INTEGER REFERENCES courses(id),
    grade      REAL
);

## 3. Adding rows: `INSERT`

In [8]:
INSERT INTO students (id, name, age, city) VALUES
    (1, 'Ada',   20, 'London'),
    (2, 'Alan',  22, 'Manchester'),
    (3, 'Grace', 21, 'New York'),
    (4, 'Linus', 19, 'Helsinki'),
    (5, 'Hedy',  23, 'Vienna');

In [9]:
INSERT INTO courses (id, title, credits) VALUES
    (1, 'Programming', 6),
    (2, 'Databases',   6),
    (3, 'Networks',    4);

In [10]:
INSERT INTO enrollments (student_id, course_id, grade) VALUES
    (1, 1, 9.5), (1, 2, 8.0),
    (2, 1, 7.0), (2, 3, 6.5),
    (3, 2, 9.0), (3, 3, 8.5),
    (4, 1, 4.5),
    (5, 2, 7.5);

## 4. Asking questions: `SELECT`

`SELECT columns FROM table` reads data. `*` means all columns.

In [11]:
SELECT * FROM students;

id,name,age,city
1,Ada,20,London
2,Alan,22,Manchester
3,Grace,21,New York
4,Linus,19,Helsinki
5,Hedy,23,Vienna


### Filtering with `WHERE` and sorting with `ORDER BY`

Combine conditions with `AND` / `OR`. `LIKE 'A%'` matches text starting with A. `ORDER BY … DESC` sorts from high to low.

In [12]:
SELECT name, age FROM students WHERE age >= 21 ORDER BY age DESC;

name,age
Hedy,23
Alan,22
Grace,21


In [13]:
SELECT name, city FROM students WHERE name LIKE 'A%' OR city = 'Vienna';

name,city
Ada,London
Alan,Manchester
Hedy,Vienna


## 5. Aggregates and `GROUP BY`

`COUNT`, `SUM`, `AVG`, `MIN`, `MAX` summarise many rows into one. `GROUP BY` makes one summary per group;
`HAVING` filters the groups.

In [14]:
SELECT COUNT(*) AS students, AVG(age) AS average_age, MIN(age) AS youngest FROM students;

students,average_age,youngest
5,21.0,19


In [15]:
SELECT course_id, COUNT(*) AS enrolled, ROUND(AVG(grade), 2) AS average_grade
FROM enrollments
GROUP BY course_id
HAVING COUNT(*) >= 2;

course_id,enrolled,average_grade
1,3,7.0
2,3,8.17
3,2,7.5


## 6. Combining tables: `JOIN`

A `JOIN` matches rows from two tables using a condition, usually a foreign key = primary key.

In [16]:
SELECT s.name, c.title, e.grade
FROM enrollments AS e
JOIN students AS s ON s.id = e.student_id
JOIN courses  AS c ON c.id = e.course_id
ORDER BY s.name, c.title;

name,title,grade
Ada,Databases,8.0
Ada,Programming,9.5
Alan,Networks,6.5
Alan,Programming,7.0
Grace,Databases,9.0
Grace,Networks,8.5
Hedy,Databases,7.5
Linus,Programming,4.5


In [17]:
SELECT c.title, ROUND(AVG(e.grade), 2) AS average_grade, COUNT(*) AS students
FROM courses AS c
JOIN enrollments AS e ON e.course_id = c.id
GROUP BY c.title
ORDER BY average_grade DESC;

title,average_grade,students
Databases,8.17,3
Networks,7.5,2
Programming,7.0,3


A `LEFT JOIN` keeps every row from the left table, even if there's no match (the missing values are `NULL`).

In [18]:
SELECT s.name, COUNT(e.course_id) AS courses
FROM students AS s
LEFT JOIN enrollments AS e ON e.student_id = s.id
GROUP BY s.name;

name,courses
Ada,2
Alan,2
Grace,2
Hedy,1
Linus,1


## 7. Changing data: `UPDATE` and `DELETE`

⚠️ Always use a `WHERE` clause, or the change applies to **every** row.

In [19]:
UPDATE enrollments SET grade = 5.0 WHERE student_id = 4 AND course_id = 1;

In [20]:
DELETE FROM enrollments WHERE grade < 7;

In [21]:
SELECT * FROM enrollments;

student_id,course_id,grade
1,1,9.5
1,2,8.0
2,1,7.0
3,2,9.0
3,3,8.5
5,2,7.5


## Try it yourself

1. List the students who live in a city that starts with `N` or `M`.
2. Show each student's name with their best grade.
3. Add a new course and enrol two students in it, then check the result with a `JOIN`.

In [22]:
SELECT name FROM students;

name
Ada
Alan
Grace
Linus
Hedy
